# Assignment

> Add blockquote



## Problem Statement:
Study analysis on the public sentiment towards a company and its stock effect their prices by extracting opinions from Twitter in determining whether public opinion has impact on the stock prices of the company.

In [ ]:
import pandas as pd
import numpy as np

path = "dataset.csv"
df = pd.read_csv(path)

# Creating a small sample for faster processing during the notebook
# df = df.head(100000).copy()

print(f"Using sample size: {len(df):,}")
print(df.head())

### Basic Descriptive Statistics

In [ ]:
print("Describe: ", df.describe())

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset="TWEET")
df.isnull().sum()

In [ ]:
print(df['LSTM_POLARITY'].unique())
print(df['TEXTBLOB_POLARITY'].unique())

In [ ]:
lstm_counts = df['LSTM_POLARITY'].value_counts()
textblob_counts = df['TEXTBLOB_POLARITY'].value_counts()

print("LSTM Distribution:\n", lstm_counts)
print("\nTextBlob Distribution:\n", textblob_counts)

In [ ]:
import sys
print(sys.executable)

In [ ]:
!{sys.executable} -m pip install matplotlib

In [ ]:
def categorize_lstm(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df['LSTM_sentiment'] = df['LSTM_POLARITY'].apply(categorize_lstm)

lstm_counts = df['LSTM_sentiment'].value_counts()
print(lstm_counts)

import matplotlib.pyplot as plt

plt.figure()
plt.pie(lstm_counts, labels=lstm_counts.index, autopct='%1.1f%%')
plt.title("LSTM Sentiment Distribution")
plt.show()

### Metada Analysis

In [ ]:
#if sentiment analysis has more neutral than +ve & -ve, there is issue in dataset
print(df.columns)
print(df['STOCK'].value_counts())


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df['cleaned_tweet'] = df['TWEET'].apply(clean_text)

In [ ]:
!{sys.executable} -m pip install nltk

In [ ]:
### Remove Stopword
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from collections import Counter

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return words

all_words = []

for tweet in df['TWEET']:
    all_words.extend(clean_text(tweet))

In [ ]:
word_counts = Counter(all_words)
top_words = word_counts.most_common(15)

words = [w[0] for w in top_words]
counts = [w[1] for w in top_words]

In [ ]:
### Plotting
import matplotlib.pyplot as plt

plt.figure()
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 15 Most Frequent Words")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
!pip install textblob

In [ ]:
from textblob import TextBlob

def get_polarity(text):
    return TextBlob(text).sentiment.polarity

df['calculated_polarity'] = df['TWEET'].apply(get_polarity)

In [ ]:
def categorize_sentiment(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['calculated_polarity'].apply(categorize_sentiment)

In [ ]:
sentiment_counts = df['sentiment'].value_counts()
print(sentiment_counts)

In [ ]:
plt.figure()
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%')
plt.title("Overall Sentiment Distribution")
plt.show()

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

df['vader_sentiment'] = df['TWEET'].apply(get_sentiment)

vader_counts = df['vader_sentiment'].value_counts()
print(vader_counts)

import matplotlib.pyplot as plt

plt.figure()
plt.pie(vader_counts, labels=vader_counts.index, autopct='%1.1f%%')
plt.title("VADER Sentiment Distribution")
plt.show()

### Preprocessing Pipeline

#### Sentence Segmentation

In [ ]:
import nltk

from nltk.tokenize import sent_tokenize

# Download the required NLTK resources
nltk.download('punkt')      # Rules for tokenization (chopping text)
nltk.download('punkt_tab')  # NEW requirement for tokenization in updated NLTK versions
nltk.download('stopwords')  # List of common, uninformative words
nltk.download('wordnet')    # Massive English dictionary for Lemmatization
nltk.download('omw-1.4')    # Supporting data for WordNet

print("NLTK Resources Downloaded Successfully!")


In [ ]:

# Sentence Segmentation
df["sentences"] = df["TWEET"].apply(sent_tokenize)



In [ ]:
for i, s in enumerate(df["sentences"].iloc[194]):
    print(f"Sentence {i+1}: {s}")

#### Handling Abbreviations

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [ ]:
# Abbreviation Dictionary
abbreviation_dict = {
    "rt": "retweet",
    "roi": "return on investment",
    "vix": "volatility index",
    "u": "you",
    "ur": "your",
    "w/": "with",
    "dm": "direct message",
    "fb": "facebook",
    "ig": "instagram",
    "li": "linkedin",
    "yt": "youtube",
    "omg": "oh my god",
    "lol": "laugh out loud"
}

#### Tokenization

In [ ]:
def tokenize_and_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    tokens = word_tokenize(text)

    expanded_tokens = []
    for word in tokens:
        expanded_word = abbreviation_dict.get(word, word)
        expanded_tokens.extend(expanded_word.split())

    clean_tokens = [word for word in expanded_tokens if word.isalpha()]
    return clean_tokens

In [ ]:
def expand_abbreviations(token_list):
  expanded_tokens = []
  for word in token_list:
    # if word in dict, replace it
    expanded_word = abbreviation_dict.get(word.lower(), word)
    expanded_tokens.append(expanded_word.split())
  return expanded_tokens

df["tokens"] = df["TWEET"].apply(tokenize_and_clean)
df["expanded_tokens"] = df["tokens"].apply(expand_abbreviations)

In [ ]:
df['expanded_tokens']

#### Stopword Removal

In [ ]:
from nltk.corpus import stopwords

# Load the English stopwords list
stop_words = set(stopwords.words('english'))

# We can also add custom stopwords specific to our dataset if we want
custom_stopwords = ['next', 'via', 'amp']
stop_words.update(custom_stopwords)

def remove_stopwords(token_list):
    # Keep the word ONLY if it is not in our stop_words list
    filtered_tokens = [word for word in token_list if word not in stop_words]
    return filtered_tokens

# Apply to our tokens
df['filtered_tokens'] = df['tokens'].apply(remove_stopwords)

# Compare the tokens before and after stopword removal
df[['tokens', 'filtered_tokens']].head()

#### Stemming or Lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Initialize our two tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def apply_stemming(token_list):
    return [stemmer.stem(word) for word in token_list]

def apply_lemmatization(token_list):
    return [lemmatizer.lemmatize(word) for word in token_list]

# Create two new columns to compare the results
df['stemmed_words'] = df['filtered_tokens'].apply(apply_stemming)
df['lemmatized_words'] = df['filtered_tokens'].apply(apply_lemmatization)

print("Stemming and Lemmatization complete!")

#### POS Tagging

In [ ]:
nltk.download('averaged_perceptron_tagger_eng')


#apply pos tag
df['pos_tags'] = df['lemmatized_words'].apply(nltk.pos_tag)



Named Entity Recognition (NER)

In [ ]:
# nltk.download('maxent_ne_chunker_tab')
# nltk.download('words')

# def apply_ner(tagged_tokens):
#   return nltk.ne_chunk(tagged_tokens)

# df['ner'] = df['pos_tags'].apply(apply_ner)

#### Word Cloud Visualization

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

all_clean_words = []
for word_list in df['lemmatized_words']:
    all_clean_words.extend(word_list)

text_for_cloud = " ".join(all_clean_words)

# Generate the word cloud
wordcloud = WordCloud(width=800, height=400,
                      background_color='white',
                      colormap='inferno',
                      max_words=100).generate(text_for_cloud)

# Plot the image
plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off") # Turn off the grid numbers
plt.title("Stock Market Tweets Word Cloud", fontsize=20, fontweight='bold')
plt.show()

pipeline comparison

In [ ]:
# Select a random row (e.g., row index 5) to inspect
row_to_inspect = 200

print("1. ORIGINAL TEXT:")
print(df['TWEET'].iloc[row_to_inspect])
print("\n2. TOKENIZED & NOISE REMOVED:")
print(df['tokens'].iloc[row_to_inspect])
print("\n3. STOPWORDS REMOVED:")
print(df['filtered_tokens'].iloc[row_to_inspect])
print("\n4. STEMMED (Chopped endings):")
print(df['stemmed_words'].iloc[row_to_inspect])
print("\n5. LEMMATIZED (Dictionary root):")
print(df['lemmatized_words'].iloc[row_to_inspect])

In [ ]:
!{sys.executable} -m pip install scikit-learn seaborn afinn

import seaborn as sns
from afinn import Afinn
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

def get_sentiment_class(score, positive_threshold=0, negative_threshold=0):
    if score > positive_threshold:
        return 'Positive'
    elif score < negative_threshold:
        return 'Negative'
    return 'Neutral'

# Reuse lemmatized words when available so TF-IDF works on cleaner tokens.
if 'lemmatized_words' in df.columns:
    df['joined_tokens'] = df['lemmatized_words'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
else:
    df['joined_tokens'] = df['TWEET'].fillna('').astype(str)

# TextBlob comparison column
df['textblob_polarity'] = df['joined_tokens'].apply(lambda text: TextBlob(text).sentiment.polarity)
df['textblob_sentiment'] = df['textblob_polarity'].apply(get_sentiment_class)

# VADER comparison column
sia = SentimentIntensityAnalyzer()
df['vader_score'] = df['TWEET'].fillna('').astype(str).apply(lambda text: sia.polarity_scores(text)['compound'])
df['vader_sentiment'] = df['vader_score'].apply(lambda score: get_sentiment_class(score, 0.05, -0.05))

# AFINN comparison column
afinn = Afinn()
df['afinn_score'] = df['joined_tokens'].apply(afinn.score)
df['afinn_sentiment'] = df['afinn_score'].apply(get_sentiment_class)

# Use the existing LSTM polarity labels to define sentiment groups for TF-IDF.
if 'LSTM_sentiment' not in df.columns and 'LSTM_POLARITY' in df.columns:
    df['LSTM_sentiment'] = df['LSTM_POLARITY'].apply(get_sentiment_class)

tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['joined_tokens'])
feature_names = tfidf_vectorizer.get_feature_names_out()
# Ensure the TF-IDF DataFrame uses the same index as the original df so .loc with df indices works
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=df.index)

sentiment_order = ['Negative', 'Neutral', 'Positive']
sentiment_columns = ['textblob_sentiment', 'vader_sentiment', 'afinn_sentiment']
sentiment_summary = pd.DataFrame({
    'TextBlob': df['textblob_sentiment'].value_counts(),
    'VADER': df['vader_sentiment'].value_counts(),
    'AFINN': df['afinn_sentiment'].value_counts(),
}).reindex(sentiment_order).fillna(0).astype(int)

tfidf_group_column = 'LSTM_sentiment' if 'LSTM_sentiment' in df.columns else 'vader_sentiment'
tfidf_top_terms = {}
for sentiment in sentiment_order:
    group_index = df.index[df[tfidf_group_column] == sentiment]
    if len(group_index) == 0:
        tfidf_top_terms[sentiment] = pd.Series(dtype=float)
        continue

    mean_scores = df_tfidf.loc[group_index].mean().sort_values(ascending=False).head(10)
    tfidf_top_terms[sentiment] = mean_scores

comparison_table = df[[
    'TWEET', 'textblob_sentiment', 'vader_sentiment', 'afinn_sentiment'
]].head(15)

print('--- Sentiment Comparison Counts ---')
display(sentiment_summary)
print('\n--- Sample Tweet-Level Comparison ---')
display(comparison_table)


In [ ]:
sns.set_theme(style='whitegrid')
pie_palette = ['#ff9999', '#aec7e8', '#98df8a']
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

method_columns = [
    ('TextBlob', 'textblob_sentiment'),
    ('VADER', 'vader_sentiment'),
    ('AFINN', 'afinn_sentiment'),
]

for ax, (method_name, column_name) in zip(axes[:3], method_columns):
    counts = df[column_name].value_counts().reindex(sentiment_order).fillna(0)
    ax.pie(
        counts,
        labels=counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=pie_palette,
        explode=(0.05, 0.05, 0.05),
        shadow=True,
    )
    ax.set_title(f'{method_name} Sentiment', fontsize=14, fontweight='bold')

tfidf_sentiment_strength = pd.Series({
    sentiment: tfidf_top_terms[sentiment].head(3).sum()
    for sentiment in sentiment_order
}).reindex(sentiment_order).fillna(0)

if tfidf_sentiment_strength.sum() > 0:
    axes[3].pie(
        tfidf_sentiment_strength,
        labels=tfidf_sentiment_strength.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=pie_palette,
        explode=(0.05, 0.05, 0.05),
        shadow=True,
    )
    axes[3].set_title(f'TF-IDF by {tfidf_group_column}', fontsize=14, fontweight='bold')
else:
    axes[3].set_title(f'TF-IDF by {tfidf_group_column}', fontsize=14, fontweight='bold')
    axes[3].text(0.5, 0.5, 'No TF-IDF data', ha='center', va='center')
    axes[3].axis('off')

plt.suptitle('Side-by-Side Comparison: TextBlob, VADER, AFINN, and TF-IDF', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

tfidf_summary_table = pd.concat({
    sentiment: pd.Series([f"{term} ({score:.3f})" for term, score in tfidf_top_terms[sentiment].items()])
    for sentiment in ['Negative', 'Neutral', 'Positive']
}, axis=1)

print('--- TF-IDF Top Terms by Sentiment Group ---')
display(tfidf_summary_table)
